# Практическая работа № 4
## Игровой цикл, Update Method и компоненты

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 4 — Паттерны: Game Loop, Update Method, Component  
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно  
**Формат сдачи:** Git-репозиторий + тесты + README + ADR  
**Максимальный балл:** 5

---

## О чём эта работа

В первых трёх модулях мы построили архитектуру игры как **набор взаимодействующих объектов**. Но всё это работает в «ручном» режиме: `Game.run()` ждёт ввода игрока и только тогда что-то делает. Мир не «живёт» сам по себе.

В этом модуле мы добавим **время** в игру:
- **Game Loop** с фиксированным шагом — сердце любой игры.
- **Update Method** — единообразное обновление всех объектов.
- **Component** — композиция поведения вместо наследования.

## Цели работы

После выполнения вы сможете:
- проектировать игровой цикл с фиксированным шагом;
- защищать цикл от спирали смерти;
- использовать аккумулятор для стабильной физики;
- выделять компоненты как dataclass;
- собирать сущности из компонентов;
- писать системы, работающие с компонентами;
- тестировать игровой цикл без запуска реального времени.

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | ветка Git, проверка тестов |
| 1 | Аудит текущего цикла | что не так с `Game.run()` |
| 2 | Проектирование игрового цикла | фиксированный шаг, аккумулятор |
| 3 | Реализация `GameLoop` | accumulator, max_steps |
| 4 | Проектирование компонентов | `Position`, `Health`, `Inventory`, `MoveSpeed` |
| 5 | Реализация `Entity` и компонентов | dataclass-компоненты |
| 6 | Реализация систем | `MovementSystem`, `TimerSystem`, `HealthSystem` |
| 7 | Интеграция с `Game` | update вызывается каждый тик |
| 8 | Юнит-тесты | детерминированное тестирование |
| 9 | Документация | ADR, UML, README |

**Важно:** этот блокнот содержит только задания. Готовые решения вы пишете сами. Если нужен образец — смотрите `practice4_game_loop.ipynb`, но **не копируйте его вслепую**.

**Для углублённого изучения теории:**
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html), [Update Method](https://gameprogrammingpatterns.com/update-method.html), [Component](https://gameprogrammingpatterns.com/component.html)
- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Подготовка

## Зачем этот шаг

Мы продолжаем в том же репозитории. Нужно убедиться, что всё работает, и создать новую ветку.

## Задание 0.1

В терминале:

```bash
git checkout main
git checkout -b module4-game-loop
pytest -v  # все тесты из модулей 1–3 должны проходить
```

**Если что-то не так** — вернитесь к модулю 3 и восстановите состояние. Не начинайте новую работу на сломанном фундаменте.

---
# Шаг 1. Аудит текущего игрового цикла

## Зачем этот шаг

Прежде чем что-то менять, нужно понять **что именно** нужно менять. Сейчас `Game.run()` — это цикл по событиям, а не по времени.

## Теория: чем плох текущий цикл

Текущий `Game.run()` (после модуля 3):

```python
while energy > 0:
    render(state)
    action = input.read_action()  # ждём ввода
    dispatch(action)
    if victory: break
```

**Четыре проблемы:**

1. **Мир не обновляется сам.** Если игрок не вводит команду — ничего не происходит. Нельзя сделать «через 5 секунд придёт Кузя».
2. **Нет времени.** Никакие таймеры, анимации, фоновые события невозможны.
3. **Нет `dt`.** Даже если добавим таймеры, они не смогут работать корректно.
4. **Нет разделения update/render.** Логика и отрисовка слиты в одну операцию.

## Задание 1.1. Аудит

Пройдитесь по своему `Game.run()` и заполните таблицу:

| Что происходит в цикле | Ответственность | Что должно измениться |
|---|---|---|
| ... | ... | ... |

**Сохраните в** `docs/analysis_module4.md`.

## Задание 1.2. Ответьте письменно

1. Почему сейчас нельзя сделать таймер («через 5 секунд...»)?
2. Какие объекты в вашей игре должны обновляться каждый тик?
3. Какие данные у этих объектов можно вынести в компоненты?
4. Какие системы нужны для обработки компонентов?

**Сохраните в** `docs/analysis_module4.md`.

---
# Шаг 2. Проектирование игрового цикла

## Зачем этот шаг

Игровой цикл — фундамент игры. Ошибка в нём проявится везде: таймеры будут неточными, физика — нестабильной, анимации — рваными.

## Теория: fixed timestep

**Fixed timestep** — это подход, при котором `update()` вызывается с **постоянным** `dt`, независимо от реального времени. Реализуется через **аккумулятор**:

```python
accumulator += frame_time
while accumulator >= fixed_dt:
    update(fixed_dt)
    accumulator -= fixed_dt
render()
```

**Зачем:** физика нелинейна. При переменном `dt` результат симуляции зависит от частоты кадров. При фиксированном — всегда одинаковый.

## Теория: спираль смерти

Если компьютер тормозит, `frame_time` растёт. Аккумулятор накапливает время, `update` вызывается всё больше раз, компьютер тормозит ещё сильнее. Игра зависает.

**Решение:** ограничить количество `update` за кадр (`max_steps`). Если не успели — **выбросить остаток времени**.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html)

## Задание 2.1. Диаграмма цикла

Нарисуйте блок-схему игрового цикла (Mermaid flowchart), включающую:

- измерение `frame_time`;
- добавление в аккумулятор;
- цикл `while accumulator >= fixed_dt`;
- ограничение `max_steps`;
- вызов `render()`;
- проверку условия выхода.

### Шаблон

```mermaid
flowchart TD
    Start([Старт]) --> Measure[Измерить frame_time]
    Measure --> Accum[accumulator += frame_time]
    Accum --> Check{accumulator >= fixed_dt?}
    Check -- Да --> Limit{steps < max_steps?}
    Limit -- Да --> Update[update(fixed_dt)]
    Update --> Sub[accumulator -= fixed_dt]
    Sub --> Check
    Limit -- Нет --> Drop[accumulator = 0]
    Drop --> Render[render()]
    Check -- Нет --> Render
    Render --> Running{Игра идёт?}
    Running -- Да --> Measure
    Running -- Нет --> End([Конец])
```

**Сохраните в** `docs/uml_module4.md`.

## Задание 2.2. Ответьте письменно

1. Почему `update` и `render` должны быть разделены?
2. Что произойдёт, если убрать `max_steps`?
3. Что произойдёт, если не обнулять аккумулятор при превышении `max_steps`?
4. Почему `fixed_dt = 1/60` — разумный выбор?

---
# Шаг 3. Реализация `GameLoop`

## Зачем этот шаг

`GameLoop` должен быть **отделён** от `Game`. `GameLoop` знает только о времени. `Game` знает о правилах игры. Это SRP.

## Теория: интерфейс GameLoop

`GameLoop` принимает две функции-колбэка:
- `update(fixed_dt)` — обновление мира;
- `render()` — отрисовка.

Такой дизайн позволяет тестировать цикл без создания игры.

## Задание 3.1. Реализовать `GameLoop`

Создайте файл `core/game_loop.py`:

**Требования:**

1. Класс `GameLoop` с конструктором:
   - `fixed_dt: float = 1/60`;
   - `max_steps: int = 5`;
   - `max_frame_time: float = 0.25` (максимум реального времени за кадр).
2. Метод `tick(frame_time, update, render)` — один проход цикла.
3. Метод `run(update, render, should_stop)` — основной цикл.
4. Свойство `accumulator` (только чтение).
5. Свойство `total_updates` — счётчик вызовов update.
6. Свойство `total_renders` — счётчик вызовов render.

**Проверьте себя:**

```python
loop = GameLoop(fixed_dt=1/60)
updates = []
loop.tick(0.020, lambda dt: updates.append(dt), lambda: None)
assert all(dt == 1/60 for dt in updates)  # все dt одинаковые
```

## Задание 3.2. Защита от спирали смерти

Убедитесь, что при очень большом `frame_time` цикл:
- не выполняет больше `max_steps` шагов;
- обнуляет аккумулятор после превышения;
- не зависает.

**Тест:**

```python
loop = GameLoop(max_steps=3)
loop.tick(10.0, update, render)  # огромный frame_time
assert loop.total_updates <= 3
```

## Задание 3.3. Коммит

```bash
git add core/game_loop.py
git commit -m "Add GameLoop with fixed timestep and death spiral protection"
```

---
# Шаг 4. Проектирование компонентов

## Зачем этот шаг

В модуле 2 мы обсуждали комбинаторный взрыв наследования. Компоненты решают эту проблему: объект **собирается** из деталей, а не наследуется от родителя.

## Теория: компонент как dataclass

**Компонент** — это dataclass, содержащий **данные** (иногда с минимальными методами). Компоненты не знают друг о друге. Логика — в системах.

**Три принципа:**
1. **Только данные.** Логика — в системах.
2. **Независимость.** Компонент не ссылается на другие компоненты.
3. **Переиспользование.** Один компонент используется у многих сущностей.

## Задание 4.1. Определить компоненты

Проанализируйте свои сущности (`Detective`, `Div`, `Location` и др.) и определите, какие данные можно вынести в компоненты.

Минимум нужно реализовать:
- `Position` — координаты (для будущей Pygame-версии).
- `Health` — здоровье/энергия.
- `Inventory` — инвентарь (улики).
- `MoveSpeed` — скорость движения.

**Дополнительно** (на выбор):
- `Timer` — таймер события.
- `Sprite` — визуальное представление.
- `AI` — параметры поведения.

## Задание 4.2. Таблица компонентов

Заполните таблицу:

| Компонент | Поля | Методы (если есть) | У каких сущностей используется |
|---|---|---|---|
| Position | x, y | move, distance_to | Detective, Div, Door |
| Health | current, maximum | damage, heal, is_alive | ... |
| Inventory | items | add, remove, size | ... |
| MoveSpeed | speed | — | ... |

**Сохраните в** `docs/uml_module4.md`.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)

---
# Шаг 5. Реализация `Entity` и компонентов

## Зачем этот шаг

**`Entity`** — это контейнер для компонентов. Он не имеет собственной логики. Поведение определяется **составом** компонентов.

## Теория: Entity как контейнер

Entity хранит компоненты в словаре `{type: instance}`. API:
- `add_component(comp)` — добавить;
- `get(ComponentType)` — получить;
- `has(ComponentType)` — проверить наличие;
- `remove(ComponentType)` — удалить.

**Почему словарь, а не атрибуты:**
- можно динамически добавлять/удалять компоненты;
- система может проверить наличие компонента через `has`;
- один компонент каждого типа (нет дублирования).

## Задание 5.1. Реализовать компоненты

Создайте файл `core/components.py`:

**Требования к `Position`:**
- `x: float = 0.0`, `y: float = 0.0`;
- метод `move(dx, dy)`;
- метод `distance_to(other) -> float`.

**Требования к `Health`:**
- `current: int = 100`, `maximum: int = 100`;
- свойство `is_alive`;
- метод `damage(amount)` с ограничением снизу (0);
- метод `heal(amount)` с ограничением сверху (maximum).

**Требования к `Inventory`:**
- `items: list = field(default_factory=list)`;
- метод `add(item)` — без дубликатов;
- метод `remove(item)`;
- свойство `size`.

**Требования к `MoveSpeed`:**
- `speed: float = 100.0`.

Все — `@dataclass`.

## Задание 5.2. Реализовать `Entity`

Создайте файл `core/entity.py`:

**Требования:**
- `__init__(name)` — имя и пустой словарь компонентов;
- `add_component(comp)` — добавить компонент;
- `get(ComponentType)` — вернуть компонент или `None`;
- `has(ComponentType) -> bool`;
- `remove(ComponentType)`;
- `__repr__` — показать имя и типы компонентов.

**Проверьте себя:**

```python
e = Entity("Аверин")
e.add_component(Position(0, 0))
e.add_component(Health(100, 100))
assert e.has(Position)
pos = e.get(Position)
pos.move(10, 20)
```

## Задание 5.3. Коммит

```bash
git add core/components.py core/entity.py
git commit -m "Add Entity and dataclass components"
```

---
# Шаг 6. Реализация систем

## Зачем этот шаг

**Система** — это класс с методом `update(entities, dt)`, который работает с компонентами. Системы не хранят состояние — они **обрабатывают** состояние.

## Теория: система как чистая логика

Каждая система:
- имеет один метод `update`;
- работает с сущностями, у которых есть **нужные компоненты**;
- не хранит состояние между вызовами.

**Запрос компонентов:** система проверяет `entity.has(Position) and entity.has(MoveSpeed)`. Только такие сущности обрабатываются.

## Задание 6.1. Реализовать `MovementSystem`

Создайте `game/systems/movement_system.py`:

**Требования:**
- класс `MovementSystem`;
- метод `update(entities, dt)`;
- для каждой сущности с `Position` и `MoveSpeed`:
  - **не двигать** сущности с `Health.current == 0` (мёртвые не ходят);
  - обновить позицию: `pos.x += speed * dt`.

**Замечание:** в текстовой игре движение — редкость. Но этот компонент нужен для Pygame-версии. Пока система может обновлять «виртуальные» позиции.

## Задание 6.2. Реализовать `TimerSystem`

Создайте `game/systems/timer_system.py`:

**Требования:**
- класс `Timer` (компонент) с полями `name`, `duration`, `callback`, `fired`;
- класс `TimerSystem`;
- метод `update(entities, dt)`;
- для каждой сущности с `Timer`:
  - уменьшить `duration` на `dt`;
  - если `duration <= 0` и таймер не сработал — вызвать `callback()` и пометить `fired = True`.

**Зачем:** таймеры позволяют делать события «через N секунд» без опроса. Пример: «через 30 секунд придёт Кузя с новостями».

## Задание 6.3. Реализовать `HealthSystem`

Создайте `game/systems/health_system.py`:

**Требования:**
- класс `HealthSystem`;
- метод `update(entities, dt)`;
- для каждой сущности с `Health.current == 0`:
  - пометить сущность как неактивную (или удалить из списка).

**Зачем:** это единая точка обработки смерти. Все сущности умирают одинаково.

## Задание 6.4. Коммит

```bash
git add game/systems/
git commit -m "Add MovementSystem, TimerSystem, HealthSystem"
```

---
# Шаг 7. Интеграция с `Game`

## Зачем этот шаг

Мы разнесли логику по системам. Теперь `Game` должен:
- иметь `GameLoop`;
- вызывать `update(dt)` каждый тик;
- вызывать `render()` каждый тик.

## Теория: интеграция без нарушения SOLID

`Game` остаётся **оркестратором**. Он не знает о компонентах — только о списке систем. Системы вызываются единообразно:

```python
def update(self, dt):
    for system in self._systems:
        system.update(self._entities, dt)
```

**Как это работает с существующим `GameSystem` (из модуля 3):**
Системы команд (`InvestigationSystem` и т.д.) обрабатывают `Action`. Системы обновления (`MovementSystem` и т.д.) обрабатывают `dt`. Это **разные типы систем** — не путайте.

Рекомендация: разделить на два списка — `command_systems` и `update_systems`.

## Задание 7.1. Обновить `Game`

В `game/game_v4.py`:

**Требования:**
- `Game` принимает `game_loop: GameLoop`;
- `Game` принимает `command_systems: list` (из модуля 3);
- `Game` принимает `update_systems: list` (новые);
- `Game` принимает `entities: list[Entity]`;
- метод `update(dt)` вызывает все `update_systems`;
- метод `run()` использует `GameLoop`.

**Упрощение:** в текстовой игре ввод блокирует цикл. Пока можно оставить `while` с вводом, но добавить вызов `update(dt)` **перед** вводом. Так таймеры будут идти.

## Задание 7.2. Проверить, что таймеры работают

Добавьте в игру таймер: «через 30 секунд после старта Кузя сообщает что-то важное». Запустите игру и проверьте, что событие срабатывает.

**Отладка:** используйте `FakeUI` и пропустите N секунд вручную, чтобы не ждать реальные 30 секунд.

## Задание 7.3. Коммит

```bash
git add game/game_v4.py
git commit -m "Integrate GameLoop and update systems into Game"
```

---
# Шаг 8. Юнит-тесты

## Зачем этот шаг

Игровой цикл и системы — критичная часть. Без тестов любая регрессия в них сломает всё. Но тестировать цикл сложно: он работает со временем.

**Решение:** детерминированное тестирование. Мы вызываем `tick(frame_time)` вручную, а не измеряем реальное время.

## Теория: как тестировать игровой цикл

**Основная идея:** `tick(frame_time)` детерминирован. Мы передаём известный `frame_time` и проверяем:
- сколько раз вызвался `update`;
- с каким `dt` вызвался;
- что аккумулятор вёл себя правильно.

**Тест на спираль смерти:** передаём огромный `frame_time` и проверяем, что `update` вызвался не больше `max_steps` раз.

## Задание 8.1. Тесты для `GameLoop`

Создайте `tests/test_game_loop.py`:

1. `test_fixed_dt` — все вызовы `update` получают одинаковый `dt`.
2. `test_accumulator` — аккумулятор накапливает и сбрасывается.
3. `test_max_steps` — при большом `frame_time` не больше `max_steps` шагов.
4. `test_accumulator_reset` — после превышения `max_steps` аккумулятор обнуляется.
5. `test_total_counts` — счётчики update и render корректны.

## Задание 8.2. Тесты для компонентов

Создайте `tests/test_components.py`:

1. `test_position_move` — позиция меняется.
2. `test_position_distance` — расстояние считается корректно.
3. `test_health_damage` — здоровье уменьшается, не уходит ниже 0.
4. `test_health_heal` — здоровье восстанавливается, не выше max.
5. `test_health_is_alive` — `is_alive` работает.
6. `test_inventory_add` — добавление без дубликатов.
7. `test_inventory_remove` — удаление работает.

## Задание 8.3. Тесты для `Entity`

Создайте `tests/test_entity.py`:

1. `test_add_component` — компонент добавляется.
2. `test_get_component` — компонент возвращается.
3. `test_get_missing` — отсутствующий компонент возвращает None.
4. `test_has_component` — проверка наличия.
5. `test_remove_component` — удаление работает.

## Задание 8.4. Тесты для систем

Создайте `tests/test_systems4.py`:

1. `test_movement_system_moves_entities` — сущности с Position и MoveSpeed двигаются.
2. `test_movement_system_skips_dead` — мёртвые не двигаются.
3. `test_movement_system_skips_static` — сущности без MoveSpeed не двигаются.
4. `test_timer_system_fires` — таймер срабатывает.
5. `test_timer_system_does_not_double_fire` — таймер не срабатывает дважды.
6. `test_health_system_deactivates_dead` — мёртвые деактивируются.

## Задание 8.5. Запустить все тесты

```bash
pytest -v
```

Все тесты (старые + новые) должны проходить.

## Задание 8.6. Коммит

```bash
git add tests/
git commit -m "Add unit tests for GameLoop, components, Entity and systems"
```

---
# Шаг 9. Документация

## Задание 9.1. Обновить README

Добавьте разделы:
- **Игровой цикл** — как он работает, что такое `fixed_dt`.
- **Компоненты** — какие есть, зачем нужны.
- **Системы** — какие есть, что обрабатывают.

## Задание 9.2. ADR-004

Создайте `docs/adr_004_game_loop.md`:

```markdown
# ADR-004: Введение Game Loop и компонентов

## Контекст
<какие проблемы были>

## Решение
<что выбрали>

## Рассмотренные альтернативы
1. <альтернатива> — почему отклонили
2. <альтернатива> — почему отклонили

## Последствия
- ✅ <плюс>
- ⚠️ <минус>
```

## Задание 9.3. Обновить UML

Добавьте в `docs/uml_module4.md`:
- диаграмму игрового цикла (flowchart);
- диаграмму классов `Entity`, компонентов, систем.

## Задание 9.4. Проверка PEP 8

```bash
pycodestyle core/ game/ entities/ ui/ tests/ main.py
```

Исправьте замечания.

## Задание 9.5. Финальный коммит

```bash
git add .
git commit -m "Document module 4: game loop, components, systems"
git checkout main
git merge module4-game-loop
```

---
# Самопроверка: чек-лист

In [ ]:
checklist = [
    "GameLoop реализован с fixed timestep",
    "Аккумулятор корректно накапливает и сбрасывает",
    "max_steps защищает от спирали смерти",
    "При превышении max_steps аккумулятор обнуляется",
    "Компоненты Position/Health/Inventory/MoveSpeed реализованы",
    "Все компоненты — dataclass",
    "Entity хранит компоненты и предоставляет get/has/remove",
    "MovementSystem двигает только сущности с Position+MoveSpeed",
    "MovementSystem не двигает мёртвых",
    "TimerSystem срабатывает один раз",
    "HealthSystem деактивирует мёртвых",
    "Game использует GameLoop и системы",
    "Таймер в игре работает",
    "Все тесты (старые + новые) проходят",
    "README обновлён",
    "ADR-004 создан",
    "UML обновлён",
    "PEP 8 без замечаний",
    "История коммитов осмысленна",
]
print("Чек-лист для самопроверки:")
for i, item in enumerate(checklist, 1):
    print(f"[ ] {i:2d}. {item}")

---
# Критерии оценки

| Критерий | Баллы | Что проверяется |
|---|---:|---|
| `GameLoop` | 1.5 | fixed_dt, аккумулятор, max_steps, защита от спирали |
| Компоненты | 1.0 | dataclass, независимость, переиспользование |
| `Entity` | 0.5 | get/has/remove, корректный API |
| Системы | 1.0 | MovementSystem, TimerSystem, HealthSystem |
| Интеграция с `Game` | 0.5 | GameLoop подключён, системы вызываются |
| Тесты | 0.5 | Все контракты покрыты, тесты детерминированы |
| **Итого** | **5.0** | |

**Штрафы:**
- Задержка: −1 балл за неделю.
- Плагиат из `practice4_game_loop.ipynb`: −2 балла.
- Отсутствие тестов на `GameLoop`: −1 балл.

**Бонусы (до +1):**
- Интерполяция для рендера.
- Дополнительные компоненты (`Sprite`, `AI`, `Timer`).
- Реализация `RenderSystem` с интерполяцией.

---
# Домашнее задание к следующему модулю

1. Завершить работу, смерджить ветку в `main`.
2. Убедиться, что проект работает локально.
3. Прочитать:
   - Nystrom R. «Game Programming Patterns», главы [State](https://gameprogrammingpatterns.com/state.html), [Strategy](https://gameprogrammingpatterns.com/strategy.html), [Command](https://gameprogrammingpatterns.com/command.html).
   - Gamma E. et al. «Design Patterns», главы про State, Strategy, Command.
4. Подготовить вопросы по темам модуля 5:
   - Конечные автоматы (FSM);
   - Паттерн State для состояний игры;
   - Strategy для взаимозаменяемых алгоритмов;
   - Command для undo/redo.

## Что дальше

В модуле 5 мы:
- добавим состояния игры (меню, расследование, бой, диалог);
- выделим стратегии поведения Кузи;
- реализуем команды игрока с возможностью отмены.